<a href="https://colab.research.google.com/github/Supan-Roy/financial-scam-analysis/blob/main/notebooks/09_external_source_generalization_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# Notebook 09: External-Source Generalization Evaluation
# Cell 1: Mount Google Drive & Verify Required Files
# ============================================================

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/financial-scam-analysis")

MODEL_DIR = PROJECT_DIR / "models" / "distilbert_binary"

EVAL_DIR = PROJECT_DIR / "data" / "external_evaluation"

# Required evaluation datasets
EVAL_FILES = {
    "external_scams": EVAL_DIR / "external_fin_scam_curated.csv",
    "primary_test": EVAL_DIR / "expanded_binary_external_test.csv",
    "balanced_test": EVAL_DIR / "balanced_binary_external_test.csv",
}

# Verify model directory
print("\n" + "=" * 60)
print("MODEL DIRECTORY")
print("=" * 60)

if MODEL_DIR.exists():
    print(f"Found: {MODEL_DIR}")

    for file in sorted(MODEL_DIR.iterdir()):
        if file.is_file():
            print(f"  ✓ {file.name}")
else:
    raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")

# Verify evaluation datasets
print("\n" + "=" * 60)
print("EVALUATION DATASETS")
print("=" * 60)

for name, path in EVAL_FILES.items():
    if path.exists():
        print(f"✓ {name}: {path.name} ({path.stat().st_size / (1024**2):.2f} MB)")
    else:
        raise FileNotFoundError(f"Missing dataset: {path}")

print("\nAll required paths verified.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

MODEL DIRECTORY
Found: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
  ✓ config.json
  ✓ model.safetensors
  ✓ tokenizer.json
  ✓ tokenizer_config.json
  ✓ training_args.bin

EVALUATION DATASETS
✓ external_scams: external_fin_scam_curated.csv (2.81 MB)
✓ primary_test: expanded_binary_external_test.csv (24.21 MB)
✓ balanced_test: balanced_binary_external_test.csv (5.34 MB)

All required paths verified.


In [10]:
# ============================================================
# Cell 2: Install & Import Dependencies
# ============================================================

%pip install -q transformers datasets accelerate scikit-learn

import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nDependencies imported successfully.")

PyTorch version: 2.14.1+cu130
CUDA available: True
Device: cuda
GPU: Tesla T4

Dependencies imported successfully.


In [11]:
# ============================================================
# Cell 3: Load Original DistilBERT Model & Tokenizer
# ============================================================

print("Loading original Notebook06 model...")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR)
)

model.to(DEVICE)
model.eval()

# Inspect model configuration
print("\n" + "=" * 60)
print("MODEL CONFIGURATION")
print("=" * 60)

print("Model path:", MODEL_DIR)
print("Architecture:", model.config.model_type)
print("Base model:", model.config._name_or_path)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)
print("Maximum positional embeddings:", model.config.max_position_embeddings)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Tokenizer vocabulary size:", tokenizer.vocab_size)

print("\nModel device:", next(model.parameters()).device)
print("Model loaded successfully.")

Loading original Notebook06 model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


MODEL CONFIGURATION
Model path: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Architecture: distilbert
Base model: /content/drive/MyDrive/financial-scam-analysis/models/distilbert_binary
Number of labels: 2
Label mapping: {0: 'LEGITIMATE', 1: 'SCAM'}
Maximum positional embeddings: 512
Tokenizer: BertTokenizer
Tokenizer vocabulary size: 30522

Model device: cuda:0
Model loaded successfully.


In [12]:
# ============================================================
# Cell 4: Load & Inspect Evaluation Datasets
# ============================================================

external_scams = pd.read_csv(EVAL_FILES["external_scams"])
primary_test = pd.read_csv(EVAL_FILES["primary_test"])
balanced_test = pd.read_csv(EVAL_FILES["balanced_test"])

datasets = {
    "External Scams": external_scams,
    "Primary Test": primary_test,
    "Balanced Test": balanced_test
}

for name, df in datasets.items():
    print("\n" + "=" * 65)
    print(name.upper())
    print("=" * 65)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nLabel distribution:")
    print(df["label"].value_counts().sort_index())

    print("\nClass proportions:")
    print(
        df["label"].value_counts(normalize=True)
        .sort_index()
        .round(4)
    )

    print("\nChannel distribution:")
    print(df["channel"].value_counts(dropna=False))

    print("\nFirst record:")
    print(df.iloc[0][["record_id", "label", "channel", "source"]])


EXTERNAL SCAMS
Shape: (792, 35)

Columns:
['record_id', 'original_text', 'normalized_text', 'label', 'channel', 'original_channel', 'is_email_gateway', 'has_gateway_headers', 'source', 'source_url', 'source_license', 'source_record_id', 'is_synthetic', 'scam_subtype', 'fraudulent_intent', 'social_engineering_tactics', 'legitimate_category', 'email_subject', 'email_body', 'duplicate_cluster_id', 'char_count_original', 'word_count_original', 'char_count_normalized', 'word_count_normalized', 'placeholder_count_original', 'placeholder_count_normalized', 'generation_scenario', 'risk_or_safety_note', 'language', 'communication_type', 'group_id', 'split', 'evaluation_origin', 'financial_relevance', 'annotation_status']

Label distribution:
label
1    792
Name: count, dtype: int64

Class proportions:
label
1    1.0
Name: proportion, dtype: float64

Channel distribution:
channel
email    522
sms      270
Name: count, dtype: int64

First record:
record_id                  EXT-SCAM-00001
label  

In [13]:
# ============================================================
# Cell 5: Evaluation Dataset Integrity Validation
# ============================================================

def validate_dataset(name, df):
    print(f"\n--- {name} ---")

    checks = {
        "Unique record IDs": df["record_id"].is_unique,
        "No missing record IDs": df["record_id"].notna().all(),
        "No missing normalized text": df["normalized_text"].notna().all(),
        "No empty normalized text": df["normalized_text"].astype(str).str.strip().ne("").all(),
        "Valid binary labels": set(df["label"].unique()).issubset({0, 1}),
    }

    for check, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {check}")

    return all(checks.values())


# Basic integrity checks
integrity_results = {
    "External Scams": validate_dataset("External Scams", external_scams),
    "Primary Test": validate_dataset("Primary Test", primary_test),
    "Balanced Test": validate_dataset("Balanced Test", balanced_test),
}

# Verify all external scam IDs are present in combined datasets
external_ids = set(external_scams["record_id"])

primary_scam_ids = set(
    primary_test.loc[primary_test["label"] == 1, "record_id"]
)

balanced_scam_ids = set(
    balanced_test.loc[balanced_test["label"] == 1, "record_id"]
)

print("\n--- External Scam Inclusion ---")

print("Primary contains all external scams:", external_ids == primary_scam_ids)
print("Balanced contains all external scams:", external_ids == balanced_scam_ids)

assert all(integrity_results.values()), "Dataset integrity check failed."
assert external_ids == primary_scam_ids
assert external_ids == balanced_scam_ids

print("\nAll integrity checks passed.")


--- External Scams ---
PASS: Unique record IDs
PASS: No missing record IDs
PASS: No missing normalized text
PASS: No empty normalized text
PASS: Valid binary labels

--- Primary Test ---
PASS: Unique record IDs
PASS: No missing record IDs
PASS: No missing normalized text
PASS: No empty normalized text
PASS: Valid binary labels

--- Balanced Test ---
PASS: Unique record IDs
PASS: No missing record IDs
PASS: No missing normalized text
PASS: No empty normalized text
PASS: Valid binary labels

--- External Scam Inclusion ---
Primary contains all external scams: True
Balanced contains all external scams: True

All integrity checks passed.


In [14]:
# ============================================================
# Cell 6: Inspect Input Lengths & Confirm Inference Settings
# ============================================================

MAX_LENGTH = 512
BATCH_SIZE = 32
SCAM_LABEL = 1
LEGITIMATE_LABEL = 0
THRESHOLD = 0.5

print("=" * 65)
print("INFERENCE CONFIGURATION")
print("=" * 65)

print("Input column:", "normalized_text")
print("Maximum sequence length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Scam label:", SCAM_LABEL)
print("Legitimate label:", LEGITIMATE_LABEL)
print("Decision threshold:", THRESHOLD)

# Inspect normalized text lengths in tokens
def inspect_token_lengths(name, df):
    sample = df["normalized_text"].astype(str).tolist()

    token_lengths = [
        len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
        for text in sample
    ]

    lengths = pd.Series(token_lengths)

    print(f"\n{name}")
    print("-" * 45)
    print("Records:", len(lengths))
    print("Mean tokens:", round(lengths.mean(), 2))
    print("Median tokens:", int(lengths.median()))
    print("95th percentile:", int(lengths.quantile(0.95)))
    print("Maximum tokens:", int(lengths.max()))
    print("Records exceeding 512:", int((lengths > MAX_LENGTH).sum()))
    print("Percentage exceeding 512:", round((lengths > MAX_LENGTH).mean() * 100, 2), "%")


inspect_token_lengths("External Scams", external_scams)
inspect_token_lengths("Primary Test", primary_test)
inspect_token_lengths("Balanced Test", balanced_test)

print("\nInput inspection completed.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (804 > 512). Running this sequence through the model will result in indexing errors


INFERENCE CONFIGURATION
Input column: normalized_text
Maximum sequence length: 512
Batch size: 32
Scam label: 1
Legitimate label: 0
Decision threshold: 0.5

External Scams
---------------------------------------------
Records: 792
Mean tokens: 265.62
Median tokens: 137
95th percentile: 1122
Maximum tokens: 5919
Records exceeding 512: 98
Percentage exceeding 512: 12.37 %

Primary Test
---------------------------------------------
Records: 7025
Mean tokens: 294.88
Median tokens: 206
95th percentile: 905
Maximum tokens: 5919
Records exceeding 512: 1182
Percentage exceeding 512: 16.83 %

Balanced Test
---------------------------------------------
Records: 1584
Mean tokens: 271.74
Median tokens: 169
95th percentile: 917
Maximum tokens: 5919
Records exceeding 512: 207
Percentage exceeding 512: 13.07 %

Input inspection completed.


In [15]:
# ============================================================
# Cell 7: Investigate Extremely Long Records
# ============================================================

def inspect_extreme_records(name, df, top_n=10):
    token_lengths = [
        len(tokenizer.encode(text, add_special_tokens=True, truncation=False))
        for text in df["normalized_text"].astype(str)
    ]

    temp = df[[
        "record_id", "label", "channel", "source",
        "char_count_normalized", "word_count_normalized"
    ]].copy()

    temp["actual_token_count"] = token_lengths

    print(f"\n{'=' * 65}")
    print(f"{name.upper()} — LONGEST RECORDS")
    print("=" * 65)

    display(
        temp.nlargest(top_n, "actual_token_count").reset_index(drop=True)
    )

    print("\nRecords exceeding 10,000 tokens:",
          int((temp["actual_token_count"] > 10000).sum()))

    print("Records exceeding 100,000 tokens:",
          int((temp["actual_token_count"] > 100000).sum()))


inspect_extreme_records("External Scams", external_scams)
inspect_extreme_records("Primary Test", primary_test)


EXTERNAL SCAMS — LONGEST RECORDS


,record_id,label,channel,source,char_count_normalized,word_count_normalized,actual_token_count
0,EXT-SCAM-00448,1,email,Jose Nazario Phishing Corpus,7357,1056,5919
1,EXT-SCAM-00348,1,email,Jose Nazario Phishing Corpus,16078,2575,3423
2,EXT-SCAM-00341,1,email,Jose Nazario Phishing Corpus,16056,2576,3402
3,EXT-SCAM-00440,1,email,Jose Nazario Phishing Corpus,3592,176,2936
4,EXT-SCAM-00502,1,email,Jose Nazario Phishing Corpus,9615,1530,2425
5,EXT-SCAM-00478,1,email,Jose Nazario Phishing Corpus,4467,269,2257
6,EXT-SCAM-00433,1,email,Jose Nazario Phishing Corpus,9599,1559,2022
7,EXT-SCAM-00298,1,email,Jose Nazario Phishing Corpus,9019,1435,2015
8,EXT-SCAM-00459,1,email,Jose Nazario Phishing Corpus,4827,514,1946
9,EXT-SCAM-00259,1,email,Jose Nazario Phishing Corpus,8019,1268,1824



Records exceeding 10,000 tokens: 0
Records exceeding 100,000 tokens: 0

PRIMARY TEST — LONGEST RECORDS


,record_id,label,channel,source,char_count_normalized,word_count_normalized,actual_token_count
0,EXT-SCAM-00448,1,email,Jose Nazario Phishing Corpus,7357,1056,5919
1,EXT-SCAM-00348,1,email,Jose Nazario Phishing Corpus,16078,2575,3423
2,EXT-SCAM-00341,1,email,Jose Nazario Phishing Corpus,16056,2576,3402
3,EXT-SCAM-00440,1,email,Jose Nazario Phishing Corpus,3592,176,2936
4,MEAJOR-LEG-035379,0,email,MeAJOR (trec6),6214,652,2549
5,MEAJOR-LEG-055175,0,email,MeAJOR (trec7),7565,457,2507
6,EXT-SCAM-00502,1,email,Jose Nazario Phishing Corpus,9615,1530,2425
7,EXT-SCAM-00478,1,email,Jose Nazario Phishing Corpus,4467,269,2257
8,MEAJOR-LEG-055583,0,email,MeAJOR (trec7),5396,623,2138
9,MEAJOR-LEG-040352,0,email,MeAJOR (trec7),4559,562,2085



Records exceeding 10,000 tokens: 0
Records exceeding 100,000 tokens: 0
